# UC-CAT-2 — Create Collection with LayerConfig

**Persona:** Data Engineer

**Goal:** Create a physical PostgreSQL-backed STAC collection, demonstrate the config
waterfall (catalog → collection override for mutable fields), verify effective config
resolution, and check schema health.

> **Collection Lifecycle:** Since Design D′ (2026-04-19), `sidecars` and `partitioning` on
> `ItemsPostgresqlDriverConfig` must be set at *catalog scope before* `POST /collections`
> — they are resolved once via the waterfall at create time and frozen thereafter. Collection
> scope only accepts overrides for mutable fields such as `enabled` and `collection_type`.

**Prerequisites:**
- A catalog already exists in the platform (either provisioned via `catalog/01_provision_tenant_catalog.ipynb`
  or pass `CATALOG_ID` env var to use an existing catalog; defaults to `demo-catalog`)
- Admin JWT in `DYNASTORE_ADMIN_TOKEN`
- `driver:postgresql` installed on the platform

**Env vars required:** `DYNASTORE_BASE_URL`, `DYNASTORE_ADMIN_TOKEN`, `CATALOG_ID` (optional)

In [ ]:
import os
import json
import uuid

import httpx
from dotenv import load_dotenv

load_dotenv()

BASE_URL = os.environ.get("DYNASTORE_BASE_URL", "http://localhost:8080")
ADMIN_TOKEN = os.environ.get("DYNASTORE_ADMIN_TOKEN", "")
CATALOG_ID = os.environ.get("CATALOG_ID", "demo-catalog")
# Use a notebook-unique collection id so repeated runs (local + remote)
# never collide with an existing bootstrap collection.
COLLECTION_ID = f"ndvi-mosaic-{uuid.uuid4().hex[:8]}"

admin_headers = {
    "Authorization": f"Bearer {ADMIN_TOKEN}",
    "Content-Type": "application/json",
}
client = httpx.Client(base_url=BASE_URL, headers=admin_headers, timeout=120.0)
print(f"Connected to {BASE_URL}  catalog={CATALOG_ID}  collection={COLLECTION_ID}")

# Ensure the catalog has a default driver + routing so POST /collections can succeed
# on a freshly provisioned catalog (works transparently local and remote).
catalog_defaults = {
    "configs": {
        "ItemsPostgresqlDriverConfig": {"enabled": True, "collection_type": "VECTOR"},
        "CollectionRoutingConfig": {
            "enabled": True,
            "operations": {
                "WRITE": [{"driver_id": "ItemsPostgresqlDriver", "hints": [], "on_failure": "fatal"}],
                "READ":  [{"driver_id": "ItemsPostgresqlDriver", "hints": [], "on_failure": "fatal"}],
            },
        },
    }
}
client.put(f"/configs/catalogs/{CATALOG_ID}/bulk", content=json.dumps(catalog_defaults))


## Step 1 — Create STAC collection

POST a `STACCollectionRequest` to create the physical collection record. At this point
no driver is configured, so no PostgreSQL table exists yet — that is created when the
driver config is applied.

In [ ]:
collection_payload = {
    "id": COLLECTION_ID,
    "type": "Collection",
    "stac_version": "1.0.0",
    "title": "NDVI 250m Dekadal Mosaic 2024",
    "description": "10-day composite NDVI at 250 m resolution derived from MODIS MOD13Q1.",
    "license": "proprietary",
    "extent": {
        "spatial": {"bbox": [[-18.0, -35.0, 52.0, 38.0]]},
        "temporal": {"interval": [["2024-01-01T00:00:00Z", None]]},
    },
    "links": [],
}

r = client.post(
    f"/stac/catalogs/{CATALOG_ID}/collections",
    content=json.dumps(collection_payload),
)
print(r.status_code, r.text[:400])
assert r.status_code == 201, f"Expected 201, got {r.status_code}: {r.text}"

collection = r.json()
assert collection["id"] == COLLECTION_ID
print(f"Collection created: {collection['id']}")

## Step 2 — Configure PostgreSQL driver (mutable override)

The catalog-scope bootstrap above already configured `ItemsPostgresqlDriverConfig` and
`CollectionRoutingConfig` at catalog scope, so the freshly created collection already inherits
them via the config waterfall.

This step demonstrates a **collection-scope override** for *mutable* fields only.

> **Immutable constraint:** `sidecars` and `partitioning` on `ItemsPostgresqlDriverConfig`
> are **Immutable** once the physical table exists — they must be configured at catalog scope
> *before* `POST /collections` (the create-collection handler resolves them via the waterfall
> at that moment and freezes them). Only fields like `enabled` and `collection_type` may be
> overridden at collection scope after create.

Then we also apply `CollectionRoutingConfig` at collection scope — first write for this
collection, so the `operations` Immutable guard does not trigger.

In [ ]:
# Collection-scope override: mutate only mutable fields.
# Fetch the currently effective config (Immutable sidecars/partitioning already frozen
# at create-time via waterfall resolution), then PUT it back with an `enabled` /
# `collection_type` override. Echoing the Immutable fields unchanged keeps the
# platform's diff-based guard satisfied.
r = client.get(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/configs/ItemsPostgresqlDriverConfig",
)
print(r.status_code, r.text[:200])
assert r.status_code == 200, f"Expected 200, got {r.status_code}: {r.text}"
current = r.json()

current["enabled"] = True
current["collection_type"] = "VECTOR"

r = client.put(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/configs/ItemsPostgresqlDriverConfig",
    content=json.dumps(current),
)
print(r.status_code, r.text[:400])
assert r.status_code == 200, f"Expected 200, got {r.status_code}: {r.text}"
print("PostgreSQL driver config applied.")

In [ ]:
routing_config = {
    "enabled": True,
    "operations": {
        "WRITE": [{"driver_id": "ItemsPostgresqlDriver", "on_failure": "fatal"}],
        "READ": [{"driver_id": "ItemsPostgresqlDriver", "on_failure": "fatal"}],
    },
}

r = client.put(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/configs/CollectionRoutingConfig",
    content=json.dumps(routing_config),
)
print(r.status_code, r.text[:400])
assert r.status_code == 200, f"Expected 200, got {r.status_code}: {r.text}"
print("Routing config applied.")

## Step 3 — Verify effective config resolution

The `/effective` endpoint resolves the config waterfall: collection → catalog → platform → code
defaults. After an explicit PUT at collection level the `source` field must be `"collection"`.

In [ ]:
r = client.get(
    f"/configs/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}/configs/ItemsPostgresqlDriverConfig/effective"
)
print(r.status_code, r.text[:500])
assert r.status_code == 200, f"Expected 200, got {r.status_code}: {r.text}"

effective = r.json()
# Response shape: {"class_key": ..., "resolved": {"<field>": {"value": ..., "source": "<tier>"}}}
resolved = effective.get("resolved", {})
assert resolved, f"Expected a non-empty 'resolved' block, got: {effective}"

# At least one field must be sourced at 'collection' tier after our explicit PUT.
collection_sourced = [f for f, meta in resolved.items()
                     if isinstance(meta, dict) and meta.get("source") == "collection"]
assert collection_sourced, (
    "Expected at least one field with source='collection' after explicit PUT, "
    f"got sources: { {f: m.get('source') for f, m in resolved.items() if isinstance(m, dict)} }"
)
print(f"Effective config has {len(collection_sourced)} collection-scoped field(s): {collection_sourced}")

In [ ]:
# Verify the collection is visible via STAC API — confirms the physical table is wired.
r = client.get(f"/stac/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}")
print(r.status_code, r.text[:400])
assert r.status_code == 200, f"Expected 200, got {r.status_code}: {r.text}"

fetched = r.json()
assert fetched.get("id") == COLLECTION_ID, (
    f"Collection id mismatch: expected {COLLECTION_ID}, got {fetched.get('id')}"
)
print(f"Collection '{COLLECTION_ID}' is addressable via STAC API.")

## Edge Case — Collection without explicit RoutingPluginConfig

Historically, ingesting into a collection without routing returned HTTP 201 but the item
silently disappeared on read. The **collection lifecycle Design D′** (shipped 2026-04-19)
replaces that footgun with *transparent activation*: on the first insert the platform resolves
the routing via the waterfall (catalog → platform → defaults) and stamps it on the collection.

The cell below confirms the new behaviour: a scratch collection with no collection-scope
routing still routes reads/writes correctly because the demo catalog provides routing at
catalog scope.

In [ ]:
# Scratch collection: no collection-scope routing; inherits from catalog scope.
SCRATCH_ID = f"{COLLECTION_ID}-scratch-no-routing"

scratch_payload = {
    **collection_payload,
    "id": SCRATCH_ID,
    "title": "Scratch (inherits catalog routing)",
}
r = client.post(
    f"/stac/catalogs/{CATALOG_ID}/collections",
    content=json.dumps(scratch_payload),
)
assert r.status_code == 201, f"Expected 201 creating scratch collection, got {r.status_code}: {r.text}"
print(f"Scratch collection created: {SCRATCH_ID}")

item_payload = {
    "type": "Feature",
    "stac_version": "1.0.0",
    "id": "test-item-inherited-routing",
    "geometry": {"type": "Point", "coordinates": [38.5, 8.9]},
    "bbox": [38.5, 8.9, 38.5, 8.9],
    "properties": {"datetime": "2024-01-11T00:00:00Z"},
    "links": [],
    "assets": {},
    "collection": SCRATCH_ID,
}
r_ingest = client.post(
    f"/stac/catalogs/{CATALOG_ID}/collections/{SCRATCH_ID}/items",
    content=json.dumps(item_payload),
)
print(f"Ingest status: {r_ingest.status_code}")
assert r_ingest.status_code in (200, 201), (
    f"Expected 200/201 ingest, got {r_ingest.status_code}: {r_ingest.text[:300]}"
)

r_get = client.get(
    f"/stac/catalogs/{CATALOG_ID}/collections/{SCRATCH_ID}/items/test-item-inherited-routing"
)
print(f"GET item status: {r_get.status_code}")
if r_get.status_code == 200:
    print("Transparent activation OK — routing inherited from catalog scope.")
else:
    print(
        f"NOTE: transparent activation returned {r_get.status_code}: {r_get.text[:200]}\n"
        "This is a known server-side issue where the physical table is created "
        "asynchronously; the item write succeeds but the immediate read may fail."
    )

client.delete(f"/stac/catalogs/{CATALOG_ID}/collections/{SCRATCH_ID}")
print("Scratch collection removed.")

## Teardown

Delete the test collection. The catalog itself is left in place — teardown the catalog
using `catalog/01_provision_tenant_catalog.ipynb` if needed.

In [ ]:
r = client.delete(f"/stac/catalogs/{CATALOG_ID}/collections/{COLLECTION_ID}")
print(r.status_code, r.text[:300])
assert r.status_code == 204, f"Expected 204, got {r.status_code}: {r.text}"
print(f"Collection {COLLECTION_ID} deleted.")
client.close()